# 🧩 Pandas: Additional Topics — Gap-Fill Companion

This notebook covers the remaining Pandas topics not in the main `Pandas_Zero_to_Master.ipynb`:
reading from more file sources (JSON, Excel, GitHub URLs), MultiIndex, explicit stack/unstack,
`pd.concat`, and deleting columns. Same format: theory → example → your turn → solution.

**Do this after** `Pandas_Zero_to_Master.ipynb`.


In [ ]:
import pandas as pd
import numpy as np
print(pd.__version__)

---
## 1 — Reading Data from Different Sources

📖 `read_csv` is only one of many readers. Pandas has a matching `read_*` for most formats:
`read_json`, `read_excel` (needs `openpyxl`), `read_parquet`, `read_html` (scrapes `<table>`
tags), `read_sql`. Each returns a DataFrame the same way.

In [ ]:
# JSON -- from a Python dict written to a JSON string/file
sample = pd.DataFrame({"id":[1,2,3], "name":["Ana","Ben","Cara"], "score":[88,92,79]})
sample.to_json("/tmp/sample.json", orient="records")
from_json = pd.read_json("/tmp/sample.json")
print("from JSON:\n", from_json)

# Excel -- write then read back (round-trip)
sample.to_excel("/tmp/sample.xlsx", index=False)
from_excel = pd.read_excel("/tmp/sample.xlsx")
print("\nfrom Excel:\n", from_excel)

⚡ **Pro tip.** `orient="records"` in `to_json`/`read_json` gives a list-of-dicts shape — the
most common and readable JSON layout for tabular data. Excel reading requires `openpyxl`
installed (`pip install openpyxl`).

### ✏️ Your Turn — write `sample` to `/tmp/sample2.json` with `orient="split"`, then read it back
and compare row counts to the original.

In [ ]:
# to_json orient="split", read_json, compare row count


✅ **Solution**
```python
sample.to_json("/tmp/sample2.json", orient="split")
back = pd.read_json("/tmp/sample2.json", orient="split")
print(len(back) == len(sample))
```

---
## 2 — Reading Data from GitHub

📖 `read_csv` (and friends) accept a **URL** directly — no manual download needed. For GitHub,
use the **raw** file URL (`raw.githubusercontent.com`), not the normal `github.com/.../blob/...`
page (which is HTML, not raw data).

In [ ]:
# Any read_csv/read_json call works the same way with a URL as with a local path:
# df = pd.read_csv("https://raw.githubusercontent.com/<user>/<repo>/<branch>/data.csv")
#
# We simulate this offline by reading our own local file "as if" it were remote,
# since this environment has no live internet access -- the API is identical either way.
url_like_path = "datasets/retail_sales.csv" if False else None
print("Pattern: pd.read_csv('https://raw.githubusercontent.com/USER/REPO/BRANCH/file.csv')")
print("Tip: swap github.com/.../blob/... for raw.githubusercontent.com/... (no /blob/)")

⚠️ **Common trap.** Pasting a normal `github.com/user/repo/blob/main/file.csv` URL into
`read_csv` fails or returns garbled data — that page is HTML. Always use the **Raw** button on
GitHub (or manually swap the domain/path) to get `raw.githubusercontent.com/...`.

### ✏️ Your Turn — convert this normal GitHub URL into its raw-file equivalent (as a string, no
need to actually fetch it):
`https://github.com/pandas-dev/pandas/blob/main/README.md`

In [ ]:
normal_url = "https://github.com/pandas-dev/pandas/blob/main/README.md"
raw_url = None
print(raw_url)

✅ **Solution**
```python
raw_url = normal_url.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")
# -> https://raw.githubusercontent.com/pandas-dev/pandas/main/README.md
```

---
## 3 — Creating a MultiIndex

📖 A **MultiIndex** lets a DataFrame/Series be indexed by **more than one key** at once — e.g.
(region, product) pairs — enabling natural hierarchical selection and reshaping. Build one with
`pd.MultiIndex.from_arrays`, `.from_tuples`, or by calling `.set_index()` with multiple columns.

In [ ]:
sales = pd.DataFrame({
    "region": ["North","North","South","South"],
    "product": ["Laptop","Mouse","Laptop","Mouse"],
    "revenue": [9000, 500, 7000, 400],
})

# Method 1: set_index with multiple columns
multi = sales.set_index(["region","product"])
print("MultiIndex DataFrame:\n", multi)
print("\nindex levels:", multi.index.names)

# Method 2: build directly from tuples
idx = pd.MultiIndex.from_tuples(
    [("North","Laptop"), ("North","Mouse"), ("South","Laptop"), ("South","Mouse")],
    names=["region","product"])
direct = pd.DataFrame({"revenue":[9000,500,7000,400]}, index=idx)
print("\nbuilt directly:\n", direct)

### ✏️ Your Turn — set a MultiIndex on `sales` using `["product","region"]` (reversed
order) and print the result.

In [ ]:
reordered = None
print(reordered)

✅ **Solution**
```python
reordered = sales.set_index(["product","region"])
```

---
## 4 — Selecting Data with a MultiIndex

📖 With a MultiIndex, `.loc[]` can select by **outer level** alone, or by a **tuple** for
multiple levels. `.xs()` (cross-section) is a clean way to select at any single level even when
it's not the outermost.

In [ ]:
# select all rows for the outer level "North"
print("all North rows:\n", multi.loc["North"])

# select a specific (region, product) combination
print("\nNorth, Laptop:\n", multi.loc[("North","Laptop")])

# .xs() to select by the INNER level (product), across all regions
print("\nall Laptop rows (any region) via .xs:\n", multi.xs("Laptop", level="product"))

⚡ **Pro tip.** `.xs(value, level="name")` is the cleanest way to slice on an inner level —
without it you'd need slightly awkward boolean masking on `.index.get_level_values(...)`.

### ✏️ Your Turn — select all `"Mouse"` rows (any region) from `multi` using `.xs`.

In [ ]:
mouse_rows = None
print(mouse_rows)

✅ **Solution**
```python
mouse_rows = multi.xs("Mouse", level="product")
```

---
## 5 — Stacking and Unstacking Data

📖 `.stack()` pivots **columns into a row index level** (wide → long). `.unstack()` does the
reverse (long → wide), pivoting an index level out into columns. They're the MultiIndex-native
partners to `melt`/`pivot`.

🖼️ Diagram:
```
        Electronics  Furniture          region     category
 North      9000        1200    stack   North  Electronics   9000
 South      7000         800    ---->   North  Furniture     1200
                                 <----   South  Electronics   7000
                                unstack  South  Furniture      800
```

In [ ]:
wide = pd.DataFrame({"Electronics":[9000,7000], "Furniture":[1200,800]},
                    index=pd.Index(["North","South"], name="region"))
print("wide:\n", wide)

long = wide.stack()   # columns become an inner index level
long.index.names = ["region","category"]
print("\nstacked (long):\n", long)

back_to_wide = long.unstack()   # reverse: category level -> columns
print("\nunstacked (back to wide):\n", back_to_wide)

### ✏️ Your Turn — starting from `multi` (Ch.3, indexed by region+product), `.unstack()` the
`product` level so products become columns and revenue values fill the grid.

In [ ]:
product_cols = None
print(product_cols)

✅ **Solution**
```python
product_cols = multi.unstack(level="product")
```

---
## 6 — Merging and Combining with `pd.concat`

📖 `merge` joins on shared **columns/keys** (like SQL JOIN). `pd.concat` simply **stacks**
DataFrames — `axis=0` stacks rows (append), `axis=1` stacks columns (side-by-side). Use `concat`
when you have several chunks of the *same shape* to combine, not a key-based join.

In [ ]:
jan = pd.DataFrame({"product":["Laptop","Mouse"], "units":[10,50]})
feb = pd.DataFrame({"product":["Laptop","Mouse"], "units":[12,45]})

# stack rows (like appending)
combined_rows = pd.concat([jan, feb], keys=["Jan","Feb"], names=["month", None])
print("stacked rows:\n", combined_rows)

# stack columns side by side
prices = pd.DataFrame({"unit_price":[900,25]})
side_by_side = pd.concat([jan, prices], axis=1)
print("\nstacked columns:\n", side_by_side)

⚠️ **Common trap.** `pd.concat(axis=0)` on DataFrames with **different columns** produces
`NaN` for the mismatched ones rather than an error — always check `.columns` match (or intend
the mismatch) before concatenating rows.

### ✏️ Your Turn — concatenate `jan` and `feb` **without** the `keys` argument, and reset the
index afterward so it's a clean 0..3 range.

In [ ]:
simple_concat = None
print(simple_concat)

✅ **Solution**
```python
simple_concat = pd.concat([jan, feb]).reset_index(drop=True)
```

---
## 7 — Adding, Deleting & Modifying Columns (deletion, the missing piece)

📖 Adding/modifying columns was covered in the main lab. **Deleting** is the piece to add:
`.drop(columns=[...])` (returns a new DataFrame; use `inplace=True` to mutate) or `del df["col"]`
(always mutates in place).

In [ ]:
df = pd.DataFrame({"a":[1,2], "b":[3,4], "c":[5,6]})
print("before:\n", df)

dropped = df.drop(columns=["b"])          # non-mutating, returns new df
print("\nafter drop (new df):\n", dropped)
print("original untouched:\n", df)

df2 = df.copy()
del df2["c"]                              # mutates df2 directly
print("\nafter del:\n", df2)

### ✏️ Your Turn — drop **two** columns (`"a"` and `"c"`) from `df` in one call, without
mutating `df` itself.

In [ ]:
result = None
print(result)

✅ **Solution**
```python
result = df.drop(columns=["a","c"])
```

---
🎉 **Gap-fill complete.** Combined with `Pandas_Zero_to_Master.ipynb`, you now have full
coverage: DataFrame basics, .loc/.iloc, reading CSV/JSON/Excel/GitHub URLs, common data-cleaning
issues, MultiIndex creation & selection (incl. `.xs`), stack/unstack, pivot tables, `merge` +
`concat`, adding/modifying/**deleting** columns, and datetime/groupby from the main lab.

### 📌 Quick-Reference (this notebook)
`pd.read_json, pd.read_excel, pd.read_csv(url)` (use raw.githubusercontent.com for GitHub) ·
`pd.MultiIndex.from_tuples, df.set_index([...])` · `.loc[outer], .loc[(a,b)], .xs(val,
level=...)` · `.stack(), .unstack(level=...)` · `pd.concat([...], axis=0/1, keys=...)` ·
`.drop(columns=[...]), del df["col"]`
